# Assignment 4: IMDB Movie Review Sentiment Analysis
### Learned Embeddings vs. Pretrained GloVe — Kaggle-ready notebook

**Goal (in plain English):** Teach a model to decide whether a movie review is **positive (1)** or **negative (0)**. Compare two ways of representing words: (A) word embeddings learned from the reviews, and (B) word embeddings pretrained on a much larger collection of text.

**Assignment rules:**
- Keep only the first **150 tokens** of each review.
- First experiment: **exactly 100 training reviews**.
- Use **exactly 10,000 validation reviews** (kept separate from training).
- Limit the vocabulary to **at most 10,000 tokens**.
- Run both embedding approaches, then increase training size to identify **the first size tested**, if any, where the learned embedding outperforms GloVe.
- Explain the role of attention and the differences between an RNN and a Transformer in your final notes, even though this notebook runs an **RNN (bidirectional LSTM)** rather than a Transformer.

> **Academic integrity:** Run this notebook yourself, check the actual results, and write your own conclusions. Results are not hard-coded. Your course requires your own work and documented output.

## Before running: attach two Kaggle datasets

From the notebook's **Add Input / Add Data** panel, attach both:

1. [IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews) — expected file: `IMDB Dataset.csv`.
2. [glove.6B.100d.txt](https://www.kaggle.com/datasets/danielwillgeorge/glove6b100dtxt) — expected file: `glove.6B.100d.txt` (directly or inside a `.zip`).

Use a **GPU accelerator** in Kaggle Settings if available. Internet access is **not required** when both datasets are attached. Then select **Run All**.

The notebook automatically locates files below `/kaggle/input`. If a file is missing, the error message explains which dataset to add.

## 1. Imports and settings
**What this does:** Loads the tools for reading data, preparing text, training an LSTM, and drawing a graph.

In [ ]:
import os, re, io, html, gc, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers

SEED = 42
MAX_TOKENS = 10_000
MAX_LENGTH = 150
EMBED_DIM = 100          # GloVe 6B 100-dimensional word vectors
BATCH_SIZE = 128
EPOCHS = 6               # Increase if you have time and results are unstable
TRAIN_SIZES = [100, 500, 2_000, 5_000, 10_000]
# Optional: add 20_000 to TRAIN_SIZES if no crossover is found.
VALIDATION_SIZE = 10_000
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPUs found:', tf.config.list_physical_devices('GPU'))
print('Training sizes:', TRAIN_SIZES)

## 2. Find the attached data files
**What this does:** Searches Kaggle's input folder. You should see the IMDB CSV and GloVe file.

In [ ]:
INPUT_DIR = Path('/kaggle/input')
if not INPUT_DIR.exists():
    raise FileNotFoundError('This notebook must run in Kaggle: /kaggle/input is missing.')

csv_files = sorted(INPUT_DIR.rglob('*.csv'))
imdb_candidates = [p for p in csv_files if ('imdb' in p.name.lower() or 'movie' in p.name.lower())]
if not imdb_candidates:
    raise FileNotFoundError('IMDB CSV missing: use Add Input and attach "IMDB Dataset of 50K Movie Reviews".')
IMDB_PATH = next((p for p in imdb_candidates if p.name.lower() == 'imdb dataset.csv'), imdb_candidates[0])

# GloVe may be provided as plain .txt or in a .zip archive.
glove_txt_files = [p for p in INPUT_DIR.rglob('*.txt') if p.name.lower() == 'glove.6b.100d.txt']
GLOVE_TEXT_PATH = glove_txt_files[0] if glove_txt_files else None
GLOVE_ZIP_PATH = None
GLOVE_ZIP_MEMBER = None
if GLOVE_TEXT_PATH is None:
    for zpath in sorted(INPUT_DIR.rglob('*.zip')):
        try:
            with zipfile.ZipFile(zpath) as archive:
                match = next((n for n in archive.namelist() if n.lower().endswith('glove.6b.100d.txt')), None)
            if match:
                GLOVE_ZIP_PATH, GLOVE_ZIP_MEMBER = zpath, match
                break
        except zipfile.BadZipFile:
            continue

if GLOVE_TEXT_PATH is None and GLOVE_ZIP_PATH is None:
    raise FileNotFoundError(
        'GloVe 100d file missing. Use Add Input and attach "glove.6B.100d.txt" '
        '(or another Kaggle dataset containing glove.6B.100d.txt).'
    )

print('IMDB file:', IMDB_PATH)
print('GloVe file:', GLOVE_TEXT_PATH or f'{GLOVE_ZIP_PATH} -> {GLOVE_ZIP_MEMBER}')

## 3. Read and understand the movie reviews
**What this does:** Reads the movie reviews and labels. A positive review is **1**; a negative review is **0**.

In [ ]:
raw_df = pd.read_csv(IMDB_PATH)
raw_df.columns = [str(c).strip().lower() for c in raw_df.columns]
if not {'review', 'sentiment'}.issubset(raw_df.columns):
    raise ValueError(f'Expected CSV columns "review" and "sentiment", got: {raw_df.columns.tolist()}')

def clean_review(text):
    """Remove HTML line breaks and decode HTML entities; vectorization comes later."""
    text = html.unescape(str(text))
    return re.sub(r'<br\s*/?>', ' ', text, flags=re.IGNORECASE)

reviews = raw_df[['review', 'sentiment']].dropna().copy()
reviews['review'] = reviews['review'].map(clean_review)
reviews['label'] = reviews['sentiment'].astype(str).str.strip().str.lower().map({'positive': 1, 'negative': 0})
reviews = reviews.dropna(subset=['label'])[['review', 'label']].copy()
reviews['label'] = reviews['label'].astype('int32')

print('Total labeled reviews:', len(reviews))
print('Positive / negative:', reviews['label'].value_counts().sort_index().to_dict())
print('Example review (first 250 characters):', reviews.iloc[0]['review'][:250])
assert len(reviews) >= VALIDATION_SIZE + max(TRAIN_SIZES), 'Dataset is too small for requested splits.'

## 4. Make a fair training/validation split
**What this does:** Randomly selects **10,000 reviews just for validation**. Training reviews come from different rows, so the model is not examined on examples used to train it.

We use the **same validation reviews for every experiment**, making the two embedding methods easier to compare. For each training size, we keep an equal number of positive and negative training examples.

In [ ]:
training_pool, val_df = train_test_split(
    reviews, test_size=VALIDATION_SIZE, random_state=SEED, stratify=reviews['label']
)
assert set(training_pool.index).isdisjoint(set(val_df.index)), 'Train and validation rows overlap!'
val_df = val_df.reset_index(drop=True)
training_pool = training_pool.reset_index(drop=True)
pos_pool = training_pool[training_pool['label'] == 1].sample(frac=1, random_state=SEED).reset_index(drop=True)
neg_pool = training_pool[training_pool['label'] == 0].sample(frac=1, random_state=SEED).reset_index(drop=True)

assert len(val_df) == 10_000

def get_balanced_training_rows(n):
    if n > 2 * min(len(pos_pool), len(neg_pool)):
        raise ValueError(f'Not enough examples for a balanced training size of {n}')
    half = n // 2
    examples = pd.concat([pos_pool.iloc[:half + n % 2], neg_pool.iloc[:half]], ignore_index=True)
    return examples.sample(frac=1, random_state=SEED + n).reset_index(drop=True)

x_val_text = val_df['review'].astype(str).tolist()
y_val = val_df['label'].to_numpy(dtype=np.float32)
print('Validation examples:', len(val_df))
print('Training pool examples:', len(training_pool))
print('Smallest training sample size:', len(get_balanced_training_rows(100)))

## 5. Load pretrained GloVe word embeddings
**Plain-English explanation:** GloVe is like a dictionary built by studying a huge amount of text. It already knows that words such as *great*, *excellent*, and *wonderful* are related.

**What this does:** Loads the 100-number representation for each word. The notebook uses this as the fixed starting dictionary for the pretrained model.

In [ ]:
def glove_lines():
    """Yield lines from either an attached .txt or a .zip containing the file."""
    if GLOVE_TEXT_PATH is not None:
        with open(GLOVE_TEXT_PATH, 'r', encoding='utf-8', errors='replace') as f:
            yield from f
    else:
        with zipfile.ZipFile(GLOVE_ZIP_PATH) as zf:
            with zf.open(GLOVE_ZIP_MEMBER, 'r') as bf:
                with io.TextIOWrapper(bf, encoding='utf-8', errors='replace') as f:
                    yield from f

glove_vectors = {}
for line in glove_lines():
    word, separator, numbers = line.rstrip('\n').partition(' ')
    if not separator:
        continue
    vector = np.fromstring(numbers, sep=' ', dtype=np.float32)
    if vector.size == EMBED_DIM:
        glove_vectors[word] = vector

print('GloVe entries loaded:', f'{len(glove_vectors):,}')
assert len(glove_vectors) > 10_000, 'GloVe file looks incomplete or is not 100d.'
print('Example vector dimensions:', glove_vectors['movie'].shape if 'movie' in glove_vectors else 'word not found')

## 6. Turn reviews into numbers
**Plain-English explanation:** AI cannot read letters directly. We convert words into numbered tokens, keeping **at most 150** per review.

We rebuild the vocabulary **using only the training rows in each experiment**. That way the 100-review experiment does not secretly learn vocabulary from thousands of additional training reviews.

In [ ]:
def prepare_text_data(train_df):
    vectorizer = layers.TextVectorization(
        max_tokens=MAX_TOKENS,
        output_mode='int',
        output_sequence_length=MAX_LENGTH,
        standardize='lower_and_strip_punctuation',
        split='whitespace',
    )
    x_train_text = train_df['review'].astype(str).tolist()
    # Learn the vocabulary from the training subset, never the validation reviews.
    vectorizer.adapt(tf.data.Dataset.from_tensor_slices(x_train_text).batch(256))
    vocab = vectorizer.get_vocabulary()
    x_train = vectorizer(tf.constant(x_train_text)).numpy().astype('int32')
    x_valid = vectorizer(tf.constant(x_val_text)).numpy().astype('int32')
    y_train = train_df['label'].to_numpy(dtype=np.float32)
    assert x_train.shape == (len(train_df), MAX_LENGTH)
    assert x_valid.shape == (VALIDATION_SIZE, MAX_LENGTH)
    assert len(vocab) <= MAX_TOKENS
    return x_train, y_train, x_valid, vocab

def make_glove_matrix(vocab):
    """Match each word in our current vocabulary with its pretrained GloVe vector."""
    matrix = np.zeros((len(vocab), EMBED_DIM), dtype=np.float32)
    covered = 0
    for idx, word in enumerate(vocab):
        if word in glove_vectors:
            matrix[idx] = glove_vectors[word]
            covered += 1
    return matrix, covered

print('Text preparation ready. Every review will have shape:', (MAX_LENGTH,))

## 7. Build the two models using the SAME LSTM architecture

**Model A — learned embedding:** Starts with a new word dictionary and learns the word representations from your available reviews.

**Model B — pretrained GloVe:** Starts with a pretrained word dictionary and keeps it fixed while the LSTM and classifier are trained. This is called **freezing the embedding layer**.

A **bidirectional LSTM** reads the review both forward and backward to learn how nearby words affect meaning. The rest of the model is the same for both approaches so we can compare the embeddings fairly.

In [ ]:
def build_lstm_model(vocab_size, pretrained_matrix=None):
    embedding_kwargs = {
        'input_dim': vocab_size,
        'output_dim': EMBED_DIM,
        'mask_zero': True,          # Ignore zero padding at the end of short reviews
        'name': 'word_embedding',
    }
    if pretrained_matrix is not None:
        embedding_kwargs['weights'] = [pretrained_matrix]
        embedding_kwargs['trainable'] = False
    else:
        embedding_kwargs['trainable'] = True

    model = tf.keras.Sequential([
        layers.Input(shape=(MAX_LENGTH,), dtype='int32'),
        layers.Embedding(**embedding_kwargs),
        layers.Bidirectional(layers.LSTM(16), name='bidirectional_lstm'),
        layers.Dropout(0.30),
        layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    return model

print('Both models share a bidirectional LSTM and classifier. Only the embedding strategy differs.')

## 8. Train both models at different training sizes

**What to expect:** This is the main experiment. It runs **two** models for each training size and records their actual accuracy on the 10,000 validation reviews.

The notebook starts with the required **100 training reviews**, then uses 500, 2,000, 5,000, and 10,000. Training can take a while, especially without a GPU. If needed, use the Kaggle **GPU accelerator** and leave this cell running until complete. Do not report results until the cell has finished.

*Important:* Small datasets make results noisy. A win at one training size does **not** establish a universal crossover point.

In [ ]:
records = []
for n in TRAIN_SIZES:
    print('\n' + '=' * 60)
    print(f'EXPERIMENT: {n:,} training reviews; {VALIDATION_SIZE:,} validation reviews')
    current_train = get_balanced_training_rows(n)
    x_train, y_train, x_valid, vocab = prepare_text_data(current_train)
    pretrained_matrix, covered = make_glove_matrix(vocab)
    print(f'Vocabulary: {len(vocab):,} / {MAX_TOKENS:,}; GloVe coverage: {covered:,} tokens')

    for model_type in ['Learned embedding', 'Pretrained GloVe (frozen)']:
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(SEED)
        use_glove = model_type.startswith('Pretrained')
        model = build_lstm_model(
            vocab_size=len(vocab),
            pretrained_matrix=pretrained_matrix if use_glove else None
        )
        callbacks = [tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=2, min_delta=0.001, restore_best_weights=True
        )]
        print('\nTraining:', model_type)
        history = model.fit(
            x_train, y_train,
            validation_data=(x_valid, y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=callbacks,
            verbose=2,
            shuffle=True,
        )
        val_loss, val_accuracy = model.evaluate(x_valid, y_val, verbose=0)
        records.append({
            'training_reviews': n,
            'model': model_type,
            'validation_accuracy': float(val_accuracy),
            'validation_loss': float(val_loss),
            'epochs_run': len(history.history['loss']),
            'vocab_size': len(vocab),
            'glove_covered_tokens': covered,
        })
        print(f'{model_type} | validation accuracy: {val_accuracy:.2%}')
        del model
        gc.collect()

results = pd.DataFrame(records)
assert len(results) == len(TRAIN_SIZES) * 2
results.to_csv(OUTPUT_DIR / 'assignment4_experiment_results.csv', index=False)
print('\nAll experiments completed; saved CSV results.')

## 9. Compare validation accuracy in a table
**How to read it:** 0.70 means the model correctly classified 70% of the held-out validation reviews. Higher is better. The **difference** column is learned minus pretrained: positive means learned embedding did better.

In [ ]:
comparison = results.pivot(
    index='training_reviews', columns='model', values='validation_accuracy'
).sort_index()
comparison['Difference: learned - GloVe'] = (
    comparison['Learned embedding'] - comparison['Pretrained GloVe (frozen)']
)
comparison['Winner'] = np.where(
    comparison['Difference: learned - GloVe'] > 0,
    'Learned embedding',
    np.where(comparison['Difference: learned - GloVe'] < 0, 'Pretrained GloVe', 'Tie')
)
print('VALIDATION ACCURACY BY TRAINING SIZE')
display(comparison.style.format({
    'Learned embedding': '{:.1%}',
    'Pretrained GloVe (frozen)': '{:.1%}',
    'Difference: learned - GloVe': '{:+.1%}'
}))
comparison.to_csv(OUTPUT_DIR / 'assignment4_comparison_table.csv')

## 10. Produce the graph needed for the assignment
**What this does:** Draws the two approaches on the same chart. The model with the higher line achieved better validation accuracy at that training size.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(comparison.index, 100 * comparison['Learned embedding'], marker='o',
        linewidth=2.5, label='Learned embedding')
ax.plot(comparison.index, 100 * comparison['Pretrained GloVe (frozen)'], marker='s',
        linewidth=2.5, label='Pretrained GloVe (frozen)')
ax.axhline(50, linestyle=':', linewidth=1, color='gray', label='Chance (50%)')
ax.set_title('IMDB Sentiment: Learned vs. Pretrained Embeddings')
ax.set_xlabel('Number of training reviews')
ax.set_ylabel('Validation accuracy (%)')
ax.set_xticks(TRAIN_SIZES)
ax.set_xticklabels([f'{x:,}' for x in TRAIN_SIZES])
ax.set_ylim(40, 100)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plot_path = OUTPUT_DIR / 'assignment4_accuracy_graph.png'
fig.savefig(plot_path, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)

## 11. Answer the assignment's main question (from ACTUAL results)

The first tested training size where learned embedding wins is a **sample-based observation**, not a guaranteed threshold for all datasets. If GloVe stays ahead at every size, that is a valid result and should be stated honestly.

In [ ]:
glove_won_at_100 = comparison.loc[100, 'Winner']
learned_wins = comparison.index[comparison['Difference: learned - GloVe'] > 0].tolist()
if learned_wins:
    crossover_message = (
        f'First TESTED training size where learned embedding beat frozen GloVe: '
        f'{min(learned_wins):,} reviews. '
        'This is not necessarily the exact crossover point between tested sizes.'
    )
else:
    crossover_message = (
        f'No crossover observed for training sizes {TRAIN_SIZES}. '
        'Try adding 20,000 reviews if you want to test a larger size.'
    )

print('Winner with exactly 100 reviews:', glove_won_at_100)
print(crossover_message)
print('Note: Your specific results can vary with the random split, seed, and training settings.')

## 12. Export a short written results summary
**What this does:** Saves a Markdown report with your actual scores. Review and expand it **in your own words** before turning it in. A final submission should explain *why* the models differ, not simply paste a table.

In [ ]:
lines = [
    '# Assignment 4 — Experiment Summary',
    '',
    '## Setup',
    '- Task: IMDB movie-review sentiment classification (positive vs. negative).',
    f'- Maximum review length: {MAX_LENGTH} tokens.',
    f'- Vocabulary cap: {MAX_TOKENS:,} tokens (learned from each training subset only).',
    f'- Validation set: {VALIDATION_SIZE:,} held-out reviews; fixed across experiments.',
    f'- Training sizes: {", ".join(f"{n:,}" for n in TRAIN_SIZES)}.',
    '- Model: bidirectional LSTM; both approaches share the same architecture.',
    '- Comparison: learned embedding versus 100d GloVe, frozen during training.',
    '',
    '## Actual validation results',
    '',
    '| Training reviews | Learned embedding | Frozen GloVe | Winner |',
    '|---:|---:|---:|---|',
]
for n, row in comparison.iterrows():
    lines.append(
        f'| {n:,} | {row["Learned embedding"]:.1%} | '
        f'{row["Pretrained GloVe (frozen)"]:.1%} | {row["Winner"]} |'
    )
lines += [
    '',
    '## Main findings',
    f'- Winner at 100 reviews: {glove_won_at_100}.',
    f'- {crossover_message}',
    '',
    '## Interpret the results in your own words',
    '- With small training data, pretrained GloVe may help because it already encodes relationships among words. This is a hypothesis to compare with your observed results, not a guaranteed outcome.',
    '- With more labeled data, a trainable embedding may learn representations tailored to the sentiment task.',
    '- Results vary because of random sampling, architecture, and training settings.',
    '',
    '## RNN versus Transformer and attention — explain in your own words',
    '- RNN/LSTM: processes a sequence through successive steps; bidirectional LSTM considers both reading directions.',
    '- Transformer: uses attention so words can relate to other words directly; adds position information to represent word order.',
    '- Self-attention example: in “The animal was tired, so it slept,” the word “it” can focus on “animal.”',
    '- This assignment IMPLEMENTS an LSTM, not a Transformer; do not claim you trained a Transformer.',
    '',
    '## Personal reflection (replace these prompts with your own work)',
    '- What surprised you about your actual accuracy table?',
    '- What limitation follows from using only 100 reviews?',
    '- Would you change the number of epochs, model type, or train sample sizes next?',
]
summary_path = OUTPUT_DIR / 'assignment4_summary.md'
summary_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(summary_path.read_text(encoding='utf-8'))
print('Saved summary:', summary_path)

## 13. Make an HTML report you can upload
**What this does:** Builds a simple HTML report containing the experiment settings, actual score table, graph, and explanation notes. It will be saved to Kaggle's working/output folder. Check and personalize your explanations before submitting.

In [ ]:
from base64 import b64encode
from html import escape as html_escape

report_rows = []
for n, row in comparison.iterrows():
    report_rows.append({
        'Training reviews': f'{n:,}',
        'Learned embedding': f'{row["Learned embedding"]:.1%}',
        'Pretrained GloVe': f'{row["Pretrained GloVe (frozen)"]:.1%}',
        'Winner': row['Winner'],
    })
report_table_html = pd.DataFrame(report_rows).to_html(index=False, border=0, classes='results')
chart_as_base64 = b64encode(plot_path.read_bytes()).decode('ascii')
report_html = f"""<!DOCTYPE html>
<html lang='en'><head><meta charset='utf-8'><title>Assignment 4 - IMDB Embeddings</title>
<style>body{{font-family:Arial,sans-serif;max-width:850px;margin:42px auto;line-height:1.55;color:#203042;padding:0 20px}}
h1,h2{{color:#173756}}table{{border-collapse:collapse;width:100%;margin:20px 0}}
th,td{{padding:11px;border:1px solid #d6e0e9;text-align:left}}th{{background:#edf3f8}}
img{{max-width:100%}}.note{{color:#536577}}</style></head>
<body><h1>Assignment 4: IMDB Review Classification</h1>
<p><strong>Method:</strong> We compared a bidirectional LSTM using newly learned 100-dimensional word embeddings
with the same LSTM using frozen pretrained GloVe 100-dimensional word embeddings.
We used 150-token reviews, a vocabulary limit of 10,000 tokens, and 10,000 fixed validation examples.
Each experiment adapted its vocabulary only on its own training reviews.</p>
<h2>Validation results</h2>{report_table_html}
<h2>Accuracy graph</h2><img alt='Validation accuracy graph' src='data:image/png;base64,{chart_as_base64}' />
<h2>Key findings</h2>
<p><strong>At 100 reviews:</strong> {html_escape(str(glove_won_at_100))} performed best.</p>
<p>{html_escape(crossover_message)}</p>
<h2>Explanation of the approaches</h2>
<p>A learned embedding is a dictionary of word representations trained on the task.
GloVe is a ready-made dictionary learned from a much larger text collection. With little training data,
the pretrained dictionary can provide useful starting information, but the observed winner is determined by the table above.</p>
<p>An LSTM reads words step by step and keeps information about earlier words. A Transformer uses attention to
connect important words across the text and adds position information to keep track of order.
This experiment used an LSTM, not a Transformer.</p>
<p class='note'><strong>Before submission:</strong> Add your own interpretation of why the actual results differ,
what you learned, and the limitations. With small data, outcomes can change with the data split and random seed.</p>
</body></html>"""
html_path = OUTPUT_DIR / 'assignment4_report.html'
html_path.write_text(report_html, encoding='utf-8')
print('HTML report saved:', html_path)


## 14. What to submit (based on the Assignment 4 PDF)

1. In Kaggle, select **Save Version** and run/save the fully executed notebook. Download your `.ipynb` file (and use the generated `assignment4_report.html` as a documented HTML output (after adding your own interpretation)). The original instructions require **Python or R code** plus documented output in **HTML, PDF, or Word**.
2. Download the outputs saved to `/kaggle/working`: `assignment4_experiment_results.csv`, `assignment4_comparison_table.csv`, `assignment4_accuracy_graph.png`, , `assignment4_summary.md`, and `assignment4_report.html`.
3. Check the numbers, explain the results in **your own words**, and include the graph/table and a written conclusion.
4. In your **existing class GitHub repository**, create a new folder named exactly **`Assignment 4`**, and upload your notebook/code and the required documented output and table/graph there.
5. Submit your repository's **`.git` URL** to Canvas. Refer to your course's Assignment Schedule for the due date.

**Suggested conclusion structure:** “At 100 reviews, ___ performed better with validation accuracy ___ versus ___. When I increased training reviews to ___, I observed ___. A possible explanation is ___. An important limitation is ___.” Fill these blanks using your own results.

**Why the model is an RNN:** The lecture's bidirectional LSTM is a sequence model, so it satisfies the assignment option to apply **RNNs or Transformers**. We chose an LSTM because the assignment also asks specifically for embedding comparisons covered in those slides.